In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
FEATURE_DIR = PROJECT_ROOT / "data" / "features"

full_features = pd.read_parquet(FEATURE_DIR / "full_features_mirrored.parquet")
print(full_features.shape)

match_ids = full_features['match_id'].unique()
print(f"Total unique matches: {len(match_ids)}")

rng = np.random.default_rng(seed=42)
rng.shuffle(match_ids)

n = len(match_ids)
train_ids = match_ids[:int(n * 0.7)]
val_ids = match_ids[int(n * 0.7):int(n * 0.85)] #validation
test_ids = match_ids[int(n * 0.85):]

print(f"Train matches: {len(train_ids)}")
print(f"Val matches: {len(val_ids)}")
print(f"Test matches: {len(test_ids)}")

train_df = full_features[full_features['match_id'].isin(train_ids)].copy()
val_df = full_features[full_features['match_id'].isin(val_ids)].copy()
test_df = full_features[full_features['match_id'].isin(test_ids)].copy()

print(f"\nTrain rows: {train_df.shape[0]}")
print(f"Val rows: {val_df.shape[0]}")
print(f"Test rows: {test_df.shape[0]}")

(178662, 27)
Total unique matches: 10000
Train matches: 7000
Val matches: 1500
Test matches: 1500

Train rows: 124866
Val rows: 26896
Test rows: 26900


In [2]:
exclude_cols = ['match_id', 'game_time_s', 'perspective', 'won', 'duration_s']
feature_cols = [c for c in full_features.columns if c not in exclude_cols]
print("Feature columns:", feature_cols)

Feature columns: ['team', 'team_net_worth_team0', 'team_kills_team0', 'team_deaths_team0', 'team_damage_team0', 'own_deaths_cumulative', 'enemy_deaths_cumulative', 'own_objectives_cumulative', 'enemy_objectives_cumulative', 'own_mid_boss_claims', 'enemy_mid_boss_claims', 'team_net_worth_team1', 'team_kills_team1', 'team_deaths_team1', 'team_damage_team1', 'net_worth_diff', 'kills_diff', 'deaths_diff', 'damage_diff', 'objectives_diff', 'deaths_cumulative_diff', 'mid_boss_diff']


In [3]:
exclude_cols = [
    'match_id', 'game_time_s', 'perspective', 'won', 'duration_s', 'team',
    'team_net_worth_team0', 'team_net_worth_team1',
    'team_kills_team0', 'team_kills_team1',
    'team_deaths_team0', 'team_deaths_team1',
    'team_damage_team0', 'team_damage_team1',
]

feature_cols = [c for c in full_features.columns if c not in exclude_cols]
print("Feature columns:", feature_cols)
print(f"\nTotal features: {len(feature_cols)}")

Feature columns: ['own_deaths_cumulative', 'enemy_deaths_cumulative', 'own_objectives_cumulative', 'enemy_objectives_cumulative', 'own_mid_boss_claims', 'enemy_mid_boss_claims', 'net_worth_diff', 'kills_diff', 'deaths_diff', 'damage_diff', 'objectives_diff', 'deaths_cumulative_diff', 'mid_boss_diff']

Total features: 13


In [4]:
import torch
from torch.nn.utils.rnn import pad_sequence

def build_sequences(df, feature_cols):
    sequences = []
    lengths = []
    labels = []
    group_keys = []

    grouped = df.sort_values('game_time_s').groupby(['match_id', 'perspective'])

    for (match_id, perspective), group in grouped:
        seq = group[feature_cols].values.astype(np.float32)
        sequences.append(torch.tensor(seq))
        lengths.append(len(seq))
        labels.append(bool(group['won'].iloc[0]))
        group_keys.append((match_id, perspective))

    padded = pad_sequence(sequences, batch_first=True, padding_value=0.0)
    lengths = torch.tensor(lengths)
    labels = torch.tensor(labels, dtype=torch.float32)

    return padded, lengths, labels, group_keys

print("Building train sequences")
X_train, len_train, y_train, keys_train = build_sequences(train_df, feature_cols)
print(f"X_train shape: {X_train.shape}")  
print(f"len_train shape: {len_train.shape}")
print(f"y_train shape: {y_train.shape}")

print("\nBuilding val sequences")
X_val, len_val, y_val, keys_val = build_sequences(val_df, feature_cols)
print(f"X_val shape: {X_val.shape}")

print("\nBuilding test sequences")
X_test, len_test, y_test, keys_test = build_sequences(test_df, feature_cols)
print(f"X_test shape: {X_test.shape}")

Building train sequences
X_train shape: torch.Size([14000, 14, 13])
len_train shape: torch.Size([14000])
y_train shape: torch.Size([14000])

Building val sequences
X_val shape: torch.Size([3000, 14, 13])

Building test sequences
X_test shape: torch.Size([3000, 14, 13])


In [5]:
MODEL_DATA_DIR = PROJECT_ROOT / "data" / "model_ready"
MODEL_DATA_DIR.mkdir(parents=True, exist_ok=True)

torch.save({
    'X_train': X_train, 'len_train': len_train, 'y_train': y_train, 'keys_train': keys_train,
    'X_val': X_val, 'len_val': len_val, 'y_val': y_val, 'keys_val': keys_val,
    'X_test': X_test, 'len_test': len_test, 'y_test': y_test, 'keys_test': keys_test,
    'feature_cols': feature_cols,
}, MODEL_DATA_DIR / "sequences.pt")

print(f"Saved to {MODEL_DATA_DIR / 'sequences.pt'}")

Saved to C:\.STUFF\coding\dl-match-analyser\data\model_ready\sequences.pt


In [6]:
import torch.nn as nn

class WinProbabilityLSTM(nn.Module):
    def __init__(self, input_size, hidden_size=32, num_layers=1):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
        )
        self.fc = nn.Linear(hidden_size, 1)

    def forward(self, x):
        # x: [batch, seq_len, input_size]
        lstm_out, _ = self.lstm(x)  # lstm_out: [batch, seq_len, hidden_size]
        logits = self.fc(lstm_out)  # [batch, seq_len, 1]
        return logits.squeeze(-1)   # [batch, seq_len]

model = WinProbabilityLSTM(input_size=len(feature_cols))
print(model)

# quick sanity check: run one batch through untrained model
sample_out = model(X_train[:4])
print("\nSample output shape:", sample_out.shape)  # should be [4, 14]
print("Sample output (untrained, meaningless yet):")
print(sample_out)


WinProbabilityLSTM(
  (lstm): LSTM(13, 32, batch_first=True)
  (fc): Linear(in_features=32, out_features=1, bias=True)
)

Sample output shape: torch.Size([4, 14])
Sample output (untrained, meaningless yet):
tensor([[ 0.3336,  0.2641, -0.1562,  0.0227,  0.0953,  0.1959, -0.0830,  0.0113,
          0.2488,  0.0731,  0.0513,  0.0428,  0.0421,  0.0408],
        [-0.0373, -0.2128, -0.1554, -0.1967,  0.0208, -0.0991, -0.1408, -0.1088,
         -0.0422, -0.0691, -0.0462, -0.0227, -0.0037,  0.0104],
        [ 0.0187, -0.0385, -0.1790, -0.1803, -0.0357, -0.1394, -0.2108, -0.1943,
         -0.1621, -0.0960, -0.0339,  0.0047,  0.0246,  0.0338],
        [-0.0753, -0.0874, -0.0893, -0.0895, -0.1564, -0.0072,  0.0229, -0.0491,
         -0.0633, -0.0439, -0.0166,  0.0040,  0.0179,  0.0268]],
       grad_fn=<SqueezeBackward1>)


In [7]:
import torch.optim as optim

def make_mask(lengths, max_len):
    # mask[i, t] = 1 if timestep t is real data for sequence i, 0 if padding
    return (torch.arange(max_len).unsqueeze(0) < lengths.unsqueeze(1)).float()

max_len = X_train.shape[1]
mask_train = make_mask(len_train, max_len)
mask_val = make_mask(len_val, max_len)

# expand y (one label per sequence) to one label per timestep
y_train_expanded = y_train.unsqueeze(1).expand(-1, max_len)
y_val_expanded = y_val.unsqueeze(1).expand(-1, max_len)

model = WinProbabilityLSTM(input_size=len(feature_cols))
optimizer = optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.BCEWithLogitsLoss(reduction='none')  # 'none' so we can apply the mask manually

def masked_loss(logits, targets, mask):
    losses = criterion(logits, targets)         # [batch, seq_len]
    losses = losses * mask                       # zero out padded positions
    return losses.sum() / mask.sum()             # average over real timesteps only

n_epochs = 20
batch_size = 128
n_train = X_train.shape[0]

for epoch in range(n_epochs):
    model.train()
    perm = torch.randperm(n_train)
    total_loss = 0.0

    for i in range(0, n_train, batch_size):
        idx = perm[i:i + batch_size]
        xb = X_train[idx]
        yb = y_train_expanded[idx]
        mb = mask_train[idx]

        optimizer.zero_grad()
        logits = model(xb)
        loss = masked_loss(logits, yb, mb)
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * len(idx)

    train_loss = total_loss / n_train

    # validation
    model.eval()
    with torch.no_grad():
        val_logits = model(X_val)
        val_loss = masked_loss(val_logits, y_val_expanded, mask_val).item()

    print(f"Epoch {epoch+1}/{n_epochs} - train_loss: {train_loss:.4f} - val_loss: {val_loss:.4f}")

Epoch 1/20 - train_loss: 0.6212 - val_loss: 0.6214
Epoch 2/20 - train_loss: 0.5975 - val_loss: 0.6163
Epoch 3/20 - train_loss: 0.5953 - val_loss: 0.6155
Epoch 4/20 - train_loss: 0.5941 - val_loss: 0.6134
Epoch 5/20 - train_loss: 0.5928 - val_loss: 0.6129
Epoch 6/20 - train_loss: 0.5921 - val_loss: 0.6115
Epoch 7/20 - train_loss: 0.5908 - val_loss: 0.6101
Epoch 8/20 - train_loss: 0.5906 - val_loss: 0.6089
Epoch 9/20 - train_loss: 0.5903 - val_loss: 0.6105
Epoch 10/20 - train_loss: 0.5885 - val_loss: 0.6036
Epoch 11/20 - train_loss: 0.5853 - val_loss: 0.6016
Epoch 12/20 - train_loss: 0.5816 - val_loss: 0.5953
Epoch 13/20 - train_loss: 0.5775 - val_loss: 0.5945
Epoch 14/20 - train_loss: 0.5693 - val_loss: 0.5807
Epoch 15/20 - train_loss: 0.5570 - val_loss: 0.5674
Epoch 16/20 - train_loss: 0.5533 - val_loss: 0.5542
Epoch 17/20 - train_loss: 0.5422 - val_loss: 0.5717
Epoch 18/20 - train_loss: 0.5356 - val_loss: 0.5785
Epoch 19/20 - train_loss: 0.5234 - val_loss: 0.5382
Epoch 20/20 - train_l

In [8]:
def compute_final_timestep_accuracy(model, X, lengths, y):
    model.eval()
    with torch.no_grad():
        logits = model(X)  
        probs = torch.sigmoid(logits)

        batch_size = X.shape[0]
        final_idx = lengths - 1  
        final_probs = probs[torch.arange(batch_size), final_idx]

        preds = (final_probs > 0.5).float()
        acc = (preds == y).float().mean().item()
    return acc, final_probs

train_acc, _ = compute_final_timestep_accuracy(model, X_train, len_train, y_train)
val_acc, _ = compute_final_timestep_accuracy(model, X_val, len_val, y_val)

print(f"Train accuracy (final timestep): {train_acc:.4f}")
print(f"Val accuracy (final timestep): {val_acc:.4f}")

Train accuracy (final timestep): 0.9159
Val accuracy (final timestep): 0.9010


In [9]:
# check: does a trivial rule (net_worth_diff > 0 => win) already get ~95%?
final_idx_train = len_train - 1
final_net_worth_diff = X_train[torch.arange(X_train.shape[0]), final_idx_train, feature_cols.index('net_worth_diff')]

trivial_preds = (final_net_worth_diff > 0).float()
trivial_acc = (trivial_preds == y_train).float().mean().item()
print(f"Trivial rule (net_worth_diff > 0) train accuracy: {trivial_acc:.4f}")

Trivial rule (net_worth_diff > 0) train accuracy: 0.9591


In [10]:
# checks percentage of win probability in time
def accuracy_by_timestep(model, X, lengths, y, feature_cols, max_len):
    model.eval()
    with torch.no_grad():
        logits = model(X)
        probs = torch.sigmoid(logits)

    results = []
    for t in range(max_len):
        valid = lengths > t  
        if valid.sum() == 0:
            continue
        preds_t = (probs[valid, t] > 0.5).float()
        y_t = y[valid]
        acc_t = (preds_t == y_t).float().mean().item()
        results.append({'timestep': t, 'n_matches': valid.sum().item(), 'accuracy': acc_t})

    return pd.DataFrame(results)

acc_by_t = accuracy_by_timestep(model, X_val, len_val, y_val, feature_cols, max_len)
print(acc_by_t)

    timestep  n_matches  accuracy
0          0       3000  0.656000
1          1       3000  0.698000
2          2       3000  0.712000
3          3       3000  0.728667
4          4       3000  0.728000
5          5       2970  0.759933
6          6       2792  0.763252
7          7       2392  0.760033
8          8       1788  0.753356
9          9       1148  0.758711
10        10        550  0.789091
11        11        190  0.752632
12        12         52  0.884615
13        13         14  0.642857


In [11]:
max_len_test = X_test.shape[1]
mask_test = make_mask(len_test, max_len_test)
y_test_expanded = y_test.unsqueeze(1).expand(-1, max_len_test)

model.eval()
with torch.no_grad():
    test_logits = model(X_test)
    test_loss = masked_loss(test_logits, y_test_expanded, mask_test).item()

print(f"Test loss: {test_loss:.4f}")

# final-timestep accuracy 
test_acc, _ = compute_final_timestep_accuracy(model, X_test, len_test, y_test)
print(f"Test accuracy (final timestep): {test_acc:.4f}")

# accuracy by time
acc_by_t_test = accuracy_by_timestep(model, X_test, len_test, y_test, feature_cols, max_len_test)
print("\nAccuracy by timestep (test set):")
print(acc_by_t_test)

Test loss: 0.5356
Test accuracy (final timestep): 0.9150

Accuracy by timestep (test set):
    timestep  n_matches  accuracy
0          0       3000  0.683333
1          1       3000  0.720333
2          2       3000  0.733000
3          3       3000  0.741333
4          4       3000  0.752000
5          5       2960  0.767905
6          6       2750  0.778182
7          7       2384  0.772232
8          8       1816  0.761013
9          9       1124  0.784698
10        10        584  0.777397
11        11        222  0.770270
12        12         58  0.793103
13        13          2  1.000000


In [12]:
MODEL_DIR = PROJECT_ROOT / "models"
MODEL_DIR.mkdir(parents=True, exist_ok=True)

torch.save({
    'model_state_dict': model.state_dict(),
    'feature_cols': feature_cols,
    'input_size': len(feature_cols),
    'hidden_size': 32,
}, MODEL_DIR / "win_probability_lstm_v1.pt")

print(f"Saved model to {MODEL_DIR / 'win_probability_lstm_v1.pt'}")

Saved model to C:\.STUFF\coding\dl-match-analyser\models\win_probability_lstm_v1.pt
